# Can we build our own widgets?

The premise of this project is that a widget's frontend is **Scala we wrote**,
compiled by Scala.js, rather than handwritten JavaScript. Everything so far has
tested the pieces around that claim. This notebook tests the claim itself.

It needs **no new code** — it inlines the bundle `./mill front.fullLinkJS`
already produces.

What is already known without a browser:

- The bundle links to a single file (`ModuleSplitStyle.FewestModules`).
- It is a valid AFM: `@JSExportTopLevel("default")` emits the factory form, and
  importing the release bundle under node yields a function returning `{ render }`.

What is **not** known, and is the whole point of this notebook: whether the VS
Code webview will accept a ~324 KB `_esm` and run Laminar inside it.

```
./mill front.fullLinkJS
./mill core.jvm.publishLocal && ./mill kernel.publishLocal
```

In [1]:
import $ivy.`io.github.quafadas::wedgie-kernel:0.1.0-SNAPSHOT`

import upickle.default.ReadWriter
import wedgie.EsmSource
import wedgie.kernel.Widget

import $ivy.$                                                 


import upickle.default.ReadWriter

import wedgie.EsmSource

import wedgie.kernel.Widget


## Load the bundle we just built

Adjust the path if your checkout is elsewhere. `fullLinkJS` rather than
`fastLinkJS`: the fast-link output is ~570 KB and not Closure-optimised, so a
failure there would not distinguish "too big" from "broken".

In [2]:
val bundlePath = java.nio.file.Paths.get(
  sys.props("user.home"), "Code", "wedgie", "out", "front", "fullLinkJS.dest", "main.js"
)

val bundle = java.nio.file.Files.readString(bundlePath)

println(s"bundle: ${bundle.length} chars (${bundle.length / 1024} KB)")
println(s"ends with: ${bundle.takeRight(60).trim}")

bundle: 332992 chars (325 KB)
ends with: { $e_default as default };
//# sourceMappingURL=main.js.map


bundlePath: Path = /Users/simon/Code/wedgie/out/front/fullLinkJS.dest/main.js
bundle: String = """'use strict';
var $p;
var $fileLevelThis = this;
var $getOwnPropertyDescriptors = (Object.getOwnPropertyDescriptors || (() => {
  var ownKeysFun;
  if ((((typeof Reflect) !== "undefined") && Reflect.ownKeys)) {
    ownKeysFun = Reflect.ownKeys;
  } else {
    var getOwnPropertySymbols = (Object.getOwnPropertySymbols || ((o) => []));
    ownKeysFun = ((o) => Object.getOwnPropertyNames(o).concat(getOwnPropertySymbols(o)));
  }
  return ((o) => {
    var ownKeys = ownKeysFun(o);
    var descriptors = ({});
    var len = (ownKeys.length | 0);
    var i = 0;
    while ((i !== len)) {
      var key = ownKeys[i];
      Object.defineProperty(descriptors, key, ({
        "configurable": true,
        "enumerable": true,
        "writable": true,
        "value": Object.getOwnPropertyDescriptor(o, key)
      }));
      i = ((i + 1) | 0);
    }
    return descriptors;
  });
})());
function $Char(c) {

The last line should show a default export. If it does not, the bundle predates
`@JSExportTopLevel("default")` — re-run `./mill front.fullLinkJS`.

## Render it

`_esm` here is the entire bundle. That is exactly the cost `EsmSource.Inline`
is documented as having, and precisely why `RemoteImport` and `CommDelivered`
exist. For a one-off check it is fine.

**Do not save this notebook afterwards** — the bundle would be serialised into
the `.ipynb`.

In [3]:
case class Smoke(ping: Int) derives ReadWriter

val smoke = Widget(Smoke(0), EsmSource.Inline(bundle))

defined class Smoke
smoke: Widget[Smoke] = wedgie.kernel.Widget@566c60ca

### Reading the result

| What you see | What it means |
| --- | --- |
| "wedgie: Laminar is mounted." and a working local counter | **The premise holds.** Scala we wrote is running in the notebook. |
| Nothing, and `Failed to load model class 'AnyModel'` in the webview console | anywidget itself did not load from the CDN — unrelated to our bundle. Run `reference.ipynb`. |
| Nothing, and a `ReferenceError`/`SyntaxError` inside the module | The bundle reached the browser but did not evaluate. A bad `_esm` fails silently from the notebook's point of view — the webview console is the only place this shows. |
| Nothing at all, anywhere | The `display_data` or `comm_open` never arrived. Check the websocket frames. |

Remember the filter: a genuine widget error always has an `ipywidgets.js` or
`ipywidgetsKernel.js` frame in its stack.

## What this does and does not prove

The counter in the rendered element is **browser-local**. `Entry.render` ignores
`model` entirely, deliberately, so that "Laminar mounts" and "sync works" stay
separable failure modes.

So a working button here proves the bundle loads and Laminar runs. It proves
nothing about state sync. The kernel half of sync is already covered by
`counter.ipynb`; joining the two is the bridge, and that is the next piece of
code to write.

In [4]:
// The kernel half still works regardless of what the frontend does with it.
smoke.set(Smoke(1))
smoke.state

res4_0: Smoke = Smoke(ping = 1)
res4_1: Smoke = Smoke(ping = 1)

## If the bundle is too large to inline

That is the outcome `probes.ipynb` exists to resolve, and it does not block the
bridge — it only decides delivery:

- probe 3 passes → `EsmSource.RemoteImport(url)`
- probes 4 and 5 pass → `EsmSource.CommDelivered(bundle)`, no host needed

Both keep `_esm` at a few hundred bytes and keep the bundle out of the `.ipynb`.